# Preencher datas e escolher filial

In [ ]:
# Data de faturamento (a partir de)
data_inicial_saida_entrega = "30/06/2026"

# Data a ser exibida no excel do protocolo de entrega
data_no_protocolo = "01/07/2026"

# Escolher filial ao trocar chaves da requisição. Altera nome do arquivo excel gerado
filial = "distribuidora"
# filial = "comercio"

# Imports

In [2]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Request API

In [ ]:
url = "https://app.omie.com.br/api/v1/produtos/nfconsultar/"

# Carrega o .env e escolhe credenciais por filial
load_dotenv()
# Procura variáveis específicas por filial, com fallback para variáveis genéricas
if filial.lower() == "distribuidora":
    app_key = os.getenv("APP_KEY_DISTRIBUIDORA") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_DISTRIBUIDORA") or os.getenv("app_secret") or os.getenv("APP_SECRET")
else:
    app_key = os.getenv("APP_KEY_COMERCIO") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_COMERCIO") or os.getenv("app_secret") or os.getenv("APP_SECRET")
print(f"Usando credenciais para filial: {filial}")

payload = {
    "call": "ListarNF",
    "app_key": app_key,
    "app_secret": app_secret,
    "param": [{
  "pagina": 1,
  "registros_por_pagina": 100,
  "ordenar_por": "CODIGO",
  "dSaiEntInicial": data_inicial_saida_entrega,
  "tpNF": 1
}]
}

response = requests.post(url, json=payload)

if response.status_code != 200:
    print(f"Erro na requisição da página 1")
else:
    data = response.json()

# Extrai os dados das notas fiscais do json e armazena em uma lista de tuplas (cnpj_cpf, numero_nf)
nfs = data.get("nfCadastro", [])
todas_nfs = []
for nf in nfs:
    numero_nf = nf.get('ide', {}).get('nNF').lstrip('0')
    cnpj_cpf = nf.get('nfDestInt', {}).get('cnpj_cpf')
    todas_nfs.append((cnpj_cpf, numero_nf))

print(todas_nfs)

Usando credenciais para filial: distribuidora
[('62.452.992/0001-45', '42607'), ('62.251.536/0001-37', '42608'), ('58.558.570/0001-81', '42609'), ('18.786.642/0004-76', '42610'), ('03.572.023/0001-69', '42611'), ('61.176.418/0001-49', '42612'), ('20.532.268/0001-81', '42613'), ('43.986.357/0001-01', '42614'), ('57.414.472/0001-08', '42615'), ('57.462.972/0001-15', '42616')]


# Algoritmo Nome Fantasia

In [4]:
#importa "clientes_concatenados_atualizada.csv" para variável df_concatenated
df_concatenated = pd.read_csv("clientes_concatenados_atualizada.csv")

#cria cnpj_col para verificar qual coluna de CNPJ existe
cnpj_col = 'cnpj_cpf' if 'cnpj_cpf' in df_concatenated.columns else 'CNPJ'

# substitui os CNPJs de todas_nfs pelos nomes fantasia usando o df_concatenated atualizado
nome_fantasia_col = 'nome_fantasia' if 'nome_fantasia' in df_concatenated.columns else 'Nome Fantasia'
cnpj_to_nome = dict(zip(df_concatenated[cnpj_col], df_concatenated[nome_fantasia_col]))
todas_nfs = [(cnpj_to_nome.get(cnpj, cnpj), num) for cnpj, num in todas_nfs]

print(f"Substituição com df_concatenated atualizado concluída. Total de NFs: {len(todas_nfs)}")
print(todas_nfs)


Substituição com df_concatenated atualizado concluída. Total de NFs: 10
[('ZDELI ITAIM', '42607'), ('ZDELI REBOUÇAS', '42608'), ('ZDELI HADDOCK', '42609'), ('ZDELI IAB - REPUBLICA', '42610'), ('SANTE BAR (ADEGA BAR)', '42611'), ('TABERNA 474 (ADEGA LANCHE)', '42612'), ('A3MA (ADEGA 728)', '42613'), ('A5HL (ADEGA A5HL)', '42614'), ('JOBA BURGER POMPEIA LTDA', '42615'), ('CEPA BAR', '42616')]


# Algoritmo Planilha Protocolo

In [5]:
# Configuração de blocos
blocos_por_planilha = 5
linhas_por_bloco = 13
linha_inicio = 7

workbook_index = 1
block_index = 0

wb = load_workbook("protocolo_entrega_alterado.xlsx")
ws = wb.active

for i in range(0, len(todas_nfs), 2):
    # Inicia um novo workbook a cada blocos_por_planilha blocos
    if block_index >= blocos_por_planilha:
        file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"

        celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
        for celula in celulas:
            nova_img = Image('logo_v2.png')
            nova_img.width = 110
            nova_img.height = 95
            ws.add_image(nova_img, celula)

        wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")

        print(f"Salvou {file_name} com {block_index} blocos")

        workbook_index += 1
        block_index = 0
        wb = load_workbook("protocolo_entrega_alterado.xlsx")
        ws = wb.active

    lote = todas_nfs[i:i+2]
    print(f"Processando lote {block_index+1} da planilha {workbook_index}: {lote}")

    row_base = linha_inicio + block_index * linhas_por_bloco
    print(f"Base row para este bloco: {row_base}")

    if len(lote) == 2:
        cnpj1, num1 = lote[0]
        cnpj2, num2 = lote[1]

        # Escreve CNPJs em pares na mesma linha
        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base, column=11, value=cnpj2)  # Coluna K

        # Escreve data_no_protocolo em pares na linha seguinte
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo)   # Coluna C
        ws.cell(row=row_base + 1, column=12, value=data_no_protocolo)  # Coluna K

        # Escreve números em pares na linha seguinte
        ws.cell(row=row_base + 3, column=4, value=num1)  # Coluna C
        ws.cell(row=row_base + 3, column=12, value=num2) # Coluna K

        print(f"Bloco {block_index+1} planilha {workbook_index}: row {row_base}/{row_base+3} -> {cnpj1},{cnpj2}/{num1},{num2}")

    elif len(lote) == 1:
        cnpj1, num1 = lote[0]

        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo) # Coluna C
        ws.cell(row=row_base + 3, column=3, value=num1) # Coluna C (seguindo o padrão de par)
        
       

        print(f"Bloco {block_index+1} planilha {workbook_index} (solitário): row {row_base}/{row_base+3} -> {cnpj1}/{num1}")

    else:
        continue

    block_index += 1

# Salva o último workbook em aberto
if block_index > 0:
    file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"
    
    celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
    for celula in celulas:
        nova_img = Image('logo_v2.png')
        nova_img.width = 110
        nova_img.height = 95
        ws.add_image(nova_img, celula)

    wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")
    print(f"Salvou {file_name} com {block_index} blocos")

Processando lote 1 da planilha 1: [('ZDELI ITAIM', '42607'), ('ZDELI REBOUÇAS', '42608')]
Base row para este bloco: 7
Bloco 1 planilha 1: row 7/10 -> ZDELI ITAIM,ZDELI REBOUÇAS/42607,42608
Processando lote 2 da planilha 1: [('ZDELI HADDOCK', '42609'), ('ZDELI IAB - REPUBLICA', '42610')]
Base row para este bloco: 20
Bloco 2 planilha 1: row 20/23 -> ZDELI HADDOCK,ZDELI IAB - REPUBLICA/42609,42610
Processando lote 3 da planilha 1: [('SANTE BAR (ADEGA BAR)', '42611'), ('TABERNA 474 (ADEGA LANCHE)', '42612')]
Base row para este bloco: 33
Bloco 3 planilha 1: row 33/36 -> SANTE BAR (ADEGA BAR),TABERNA 474 (ADEGA LANCHE)/42611,42612
Processando lote 4 da planilha 1: [('A3MA (ADEGA 728)', '42613'), ('A5HL (ADEGA A5HL)', '42614')]
Base row para este bloco: 46
Bloco 4 planilha 1: row 46/49 -> A3MA (ADEGA 728),A5HL (ADEGA A5HL)/42613,42614
Processando lote 5 da planilha 1: [('JOBA BURGER POMPEIA LTDA', '42615'), ('CEPA BAR', '42616')]
Base row para este bloco: 59
Bloco 5 planilha 1: row 59/62 -> J